# Feature Ranking Workbench

This notebook ranks raw candidate features from the extracted per-sample CSV. It does **not** assume a fixed difficulty score.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path('/fs/nexus-projects/pc_driving/yaghoubi/tail-risk-motion-prediction')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.feature_analysis import candidate_feature_columns, summarize_feature_candidates, transform_series

sns.set_theme(style='whitegrid')
ARTIFACTS_ROOT = REPO_ROOT / 'artifacts'

def resolve_artifact_root(name: str) -> Path:
    path = ARTIFACTS_ROOT / name
    if not path.exists():
        raise FileNotFoundError(f'Artifact root not found: {path}')
    return path


In [ ]:
CONFIG = {
    'artifact_root': 'day2_train20',
    'dataset': 'av2',
    'split': 'train',
    'error_column': 'mtr_minfde6',
    'top_k': 20,
    'selected_feature': 'kalman_difficulty_6s',
    'selected_transform': 'raw',
    'hist_bins': 60,
}


In [ ]:
ARTIFACT_ROOT = resolve_artifact_root(CONFIG['artifact_root'])
TABLE_ROOT = ARTIFACT_ROOT / 'tables'
METRIC_ROOT = ARTIFACT_ROOT / 'metrics'
table_path = TABLE_ROOT / f"{CONFIG['dataset']}_{CONFIG['split']}_feature_table.csv"
df = pd.read_csv(table_path)
preferred_errors = ['mtr_minfde6', 'mtr_brier_minfde6', 'mtr_minade6', 'cv_fde', 'cv_ade']
if CONFIG['error_column'] not in df.columns:
    CONFIG['error_column'] = next((c for c in preferred_errors if c in df.columns), CONFIG['error_column'])
summary = summarize_feature_candidates(df, dataset=CONFIG['dataset'], split=CONFIG['split'])
current = summary[summary['error_column'] == CONFIG['error_column']].copy()
display(Markdown(f"**Artifact root:** `{CONFIG['artifact_root']}`  \
**Table:** `{table_path.name}`  \
**Rows:** {len(df):,}  \
**Candidate features:** {len(candidate_feature_columns(df))}  \
**Active error column:** `{CONFIG['error_column']}`"))
display(current.head(CONFIG['top_k']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_df = current.head(CONFIG['top_k']).copy()
sns.barplot(data=plot_df, y='score_name', x='spearman', ax=axes[0], color='#4c78a8')
axes[0].set_title(f"Top {CONFIG['top_k']} by Spearman")
sns.barplot(data=plot_df, y='score_name', x='top20_error_capture', ax=axes[1], color='#f58518')
axes[1].axvline(0.2, ls='--', c='k', lw=1)
axes[1].set_title(f"Top {CONFIG['top_k']} by Top-20% Error Capture")
plt.tight_layout()
plt.show()


In [ ]:
score_col = transform_series(df[CONFIG['selected_feature']], CONFIG['selected_transform'])
plot_df = pd.DataFrame({
    'score': score_col,
    'error': pd.to_numeric(df[CONFIG['error_column']], errors='coerce'),
}).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(plot_df['score'], bins=CONFIG['hist_bins'])
axes[0].set_title(f"{CONFIG['selected_feature']} ({CONFIG['selected_transform']})")
axes[1].scatter(plot_df['score'], plot_df['error'], s=8, alpha=0.25)
axes[1].set_xlabel('score')
axes[1].set_ylabel(CONFIG['error_column'])
axes[1].set_title(f"{CONFIG['selected_feature']} vs {CONFIG['error_column']}")
plt.tight_layout()
plt.show()


In [ ]:
error_cols = [c for c in ['mtr_minade6', 'mtr_minfde6', 'mtr_brier_minfde6', 'cv_ade', 'cv_fde'] if c in df.columns]
wide = summary[summary['score_name'].isin(current['score_name'].head(CONFIG['top_k']))]
pivot = wide.pivot_table(index='score_name', columns='error_column', values='spearman')
display(pivot.sort_values(CONFIG['error_column'], ascending=False).head(CONFIG['top_k']))
